# 🚀 NanoVector: Bare-Metal Vector Search & Episodic Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminsk/nanovector/blob/main/notebooks/nanovector_quickstart.ipynb)
[![GitHub stars](https://img.shields.io/github/stars/eminsk/nanovector?style=social)](https://github.com/eminsk/nanovector)
[![PyPI](https://img.shields.io/pypi/v/nanovector.svg)](https://pypi.org/project/nanovector/)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

**NanoVector** is a bare-metal, SIMD-accelerated vector search and episodic memory engine engineered in ~120KB of pure C99/AVX2/NEON/FASM for AI agents, edge devices, and local LLM RAG applications.

### 🌟 Key Highlights
- **Zero Bloat:** Compiles in < 3 seconds, zero heavy dependencies (unlike Chroma/FAISS).
- **Bare-Metal SIMD:** Unrolled AVX2 + FMA kernels (32 floats/iteration) & ARM NEON.
- **Microsecond Latency:** Sub-0.05 ms search time on CPU.
- **Agent Memory Ready:** Built-in episodic memory persistence with single-file `.nvec` storage.
- **Zero-Copy Batch Ingestion:** Ingest 1,000,000+ vectors/second directly from NumPy arrays.

## 1. 📦 Installation

Install `nanovector` directly from PyPI (compiles in seconds on Colab with native AVX2 SIMD acceleration):

In [ ]:
# Install NanoVector and numpy
!pip install -q nanovector

# Optionally install sentence-transformers for real semantic search demo
!pip install -q sentence-transformers

## 2. ⚡ Verification & SIMD Backend Detection

Verify the installation and inspect the active hardware acceleration backend (AVX2/FMA on x86_64 Colab CPU):

In [ ]:
import nanovector

print(f"NanoVector Version: v{nanovector.version()}")
print(f"Active SIMD Backend: {nanovector.simd_backend()}")

## 3. 🎯 10-Line Quickstart: Cosine Similarity Vector Index

Create an index, insert embeddings with metadata, and perform top-$K$ nearest-neighbor search:

In [ ]:
import numpy as np
from nanovector import Index

# 1. Initialize an index for 384-dimensional embeddings (e.g. all-MiniLM-L6-v2)
dim = 384
index = Index(dim=dim, metric="cosine")

# 2. Add sample vectors with JSON metadata
rng = np.random.default_rng(42)
sample_vectors = rng.standard_normal((5, dim)).astype(np.float32)
doc_ids = ["doc_0", "doc_1", "doc_2", "doc_3", "doc_4"]
metas = [
    '{"category": "physics", "title": "Quantum Mechanics"}',
    '{"category": "ai", "title": "Autonomous Agents"}',
    '{"category": "history", "title": "Roman Empire"}',
    '{"category": "ai", "title": "Vector Databases"}',
    '{"category": "space", "title": "James Webb Telescope"}'
]

index.add_batch(doc_ids, sample_vectors, metas)
print(f"Total documents indexed: {len(index)}")

# 3. Query with target vector (exact match with doc_1)
query_vec = sample_vectors[1]
results = index.search(query_vec, top_k=3)

print("\n--- Top Search Results ---")
for rank, match in enumerate(results, 1):
    print(f"#{rank} ID: {match.id:<8} Score: {match.score:.4f} | Meta: {match.metadata}")

## 4. 🧠 AI Agent Episodic Memory in Action

Build a persistent episodic memory system for an AI Agent.
The agent recalls relevant past conversations, instructions, and user preferences based on real semantic text embeddings!

In [ ]:
from sentence_transformers import SentenceTransformer
import json

# Load a lightweight, high-performance embedding model (~80MB)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create an episodic memory index
memory = Index(dim=384, metric="cosine")

# Simulated agent experiences / dialogue history
experiences = [
    {"id": "turn_001", "text": "The user's name is Alex and prefers Python with type hints.", "type": "user_preference"},
    {"id": "turn_002", "text": "Discussed setting up PostgreSQL on AWS RDS with connection pooling.", "type": "task_history"},
    {"id": "turn_003", "text": "User ordered a mechanical keyboard with tactile brown switches.", "type": "personal_fact"},
    {"id": "turn_004", "text": "Debugged a PyTorch CUDA out of memory error using gradient accumulation.", "type": "troubleshooting"},
    {"id": "turn_005", "text": "User lives in New York and works in the Eastern Time zone (EST).", "type": "user_preference"},
    {"id": "turn_006", "text": "Designed a SIMD-accelerated vector search engine in pure C.", "type": "project_notes"},
]

# Encode and store into episodic memory
texts = [exp["text"] for exp in experiences]
embeddings = model.encode(texts, normalize_embeddings=True).astype(np.float32)
ids = [exp["id"] for exp in experiences]
metas = [json.dumps(exp) for exp in experiences]

memory.add_batch(ids, embeddings, metas)
print(f"Stored {len(memory)} episodic memories into agent brain.\n")

# Agent recall function
def agent_recall(prompt: str, top_k: int = 2):
    q_vec = model.encode([prompt], normalize_embeddings=True)[0].astype(np.float32)
    matches = memory.search(q_vec, top_k=top_k)
    print(f"🧠 Query Prompt: '{prompt}'")
    print("   Recalled Memories:")
    for m in matches:
        data = json.loads(m.metadata)
        print(f"   - [score: {m.score:.4f}] {data['text']}")
    print()

# Test semantic episodic recall
agent_recall("What programming language and code style does the user like?")
agent_recall("How did we resolve the GPU VRAM error?")
agent_recall("What time zone is the user located in?")

## 5. 💾 Single-File Persistence (`.nvec`)

Save the agent's brain to a compact binary `.nvec` file and reload it instantly with zero overhead:

In [ ]:
import os

# Save the index to disk
nvec_path = "agent_brain.nvec"
memory.save(nvec_path)

file_size_kb = os.path.getsize(nvec_path) / 1024
print(f"Saved memory index to '{nvec_path}' ({file_size_kb:.2f} KB)")

# Load the memory index in a fresh instance
restored_memory = nanovector.load(nvec_path)
print(f"Successfully restored index! Vectors: {len(restored_memory)}, Dim: {restored_memory.dim}")

# Verify recall on restored index
test_q = model.encode(["Alex coding preferences"])[0].astype(np.float32)
top_match = restored_memory.search(test_q, top_k=1)[0]
print(f"Verification query result: {top_match.metadata}")

## 6. 🚀 Extreme High-Throughput Batch Ingestion & Search Benchmark

Benchmark NanoVector's raw SIMD speed directly on this Colab VM:
- Ingest **50,000 vectors** (384-dim) via Zero-Copy NumPy buffer protocol
- Measure insertion throughput (vectors/sec)
- Measure search latency (microseconds per query) and QPS

In [ ]:
import time

N_BENCH = 50000
DIM_BENCH = 384

print(f"Generating {N_BENCH:,} random {DIM_BENCH}-dim vectors...")
rng = np.random.default_rng(42)
bench_matrix = rng.standard_normal((N_BENCH, DIM_BENCH)).astype(np.float32)
bench_ids = [f"item_{i:06d}" for i in range(N_BENCH)]

bench_index = Index(dim=DIM_BENCH, metric="cosine")

# Measure Batch Ingestion Rate
t0 = time.perf_counter()
bench_index.add_batch(bench_ids, bench_matrix)
t_ingest = time.perf_counter() - t0
ingest_rate = N_BENCH / t_ingest

print(f"⚡ Batch Ingested {N_BENCH:,} vectors in {t_ingest*1000:.2f} ms")
print(f"⚡ Ingestion Rate: {ingest_rate:,.0f} vectors/second!")

# Measure Search Latency & QPS
NUM_QUERIES = 200
query_vectors = rng.standard_normal((NUM_QUERIES, DIM_BENCH)).astype(np.float32)

t0 = time.perf_counter()
for q in query_vectors:
    bench_index.search(q, top_k=5)
t_search = time.perf_counter() - t0

avg_latency_ms = (t_search / NUM_QUERIES) * 1000.0
avg_latency_us = avg_latency_ms * 1000.0
qps = NUM_QUERIES / t_search

print(f"\n⚡ Search Across {N_BENCH:,} vectors:")
print(f"⚡ Average Latency: {avg_latency_ms:.3f} ms ({avg_latency_us:.1f} µs) per query")
print(f"⚡ Query Throughput: {qps:,.0f} QPS on single thread!")

## 7. 🔗 Summary & Resources

NanoVector provides bare-metal vector search with **no overhead, no daemons, and no bloat**:

- **GitHub Repository:** [https://github.com/eminsk/nanovector](https://github.com/eminsk/nanovector)
- **PyPI:** [https://pypi.org/project/nanovector/](https://pypi.org/project/nanovector/)
- **Documentation & Architecture:** [NanoVector Readme](https://github.com/eminsk/nanovector#architecture)

⭐️ If you find NanoVector useful for your AI agents and edge applications, please star the project on GitHub!